# App data generation
This notebook is used to serialise waveform and model output data into .pkl format for use by the dashboard

In [ ]:
import seisbench.data as sbd
import seisbench.util as sbu
import seisbench.generate as sbg
import seisbench.models as sbm
from seisbench.util import worker_seeding

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
from obspy import UTCDateTime

import os, obspy, sys, pickle
from pathlib import Path
from tqdm import tqdm
import pandas as pd
import vis.app_utils as autils

PROJ_ROOT = "FILL IN UR PROJECT ROOT"
# RUN THIS IS PROJECT ROOT I.E. eqcomp/
os.chdir(PROJ_ROOT)

import utils.config as cfg
import utils.eval as eutils
import utils.model as mutils
import utils.train as tutils

# Load pre-trained model from seisbench Model weights are in fp32 (float)
configs = {"EQTransformer": mutils.EQTransformerConfig(),
           "EQCCTP": mutils.EQCCTConfig("P"),
           "EQCCTS": mutils.EQCCTConfig("S"),
           "PhaseNet": mutils.PhaseNetConfig()
        } # Add here

In [68]:
def build_sample_object(sample, metadata, models, model_names, threshold=0.2):
    """
    Build a unified sample object containing:
    - waveform
    - ground truth labels
    - true picks
    - predictions from multiple models
    - predicted picks
    - errors (in seconds)
    - TP/FP/FN/TN classification
    - merged EQCCT (EQCCTP + EQCCTS)
    """

    # 1. Extract waveform + ground truth
    X = sample["X"]              # shape (3, T)
    y_true = sample["y"]         # shape (3, T)

    # 2. Extract true picks from metadata
    true_picks = {
        "P": metadata.get("trace_p_arrival_sample", None),
        "S": metadata.get("trace_s_arrival_sample", None),
    }

    # 3. Extract event type
    event_type = metadata.get("event_type", "unknown")

    # 4. Noise → override true picks
    if event_type == "noise":
        true_picks = {"P": None, "S": None}

    # Unified sample object
    obj = {
        "index": metadata.get("index", None),
        "X": X,
        "y_true": y_true,
        "true_picks": true_picks,
        "event_type": event_type,
        "source_id": metadata.get("source_id", None),
        "source_origin_time": str(metadata.get("source_origin_time", None)),
        "source_depth_km": metadata.get("source_depth_km", None),
        "source_magnitude": metadata.get("source_magnitude", None),
        "station_network_code": metadata.get("station_network_code", None),
        "station_code": metadata.get("station_code", None),
        "trace_channel": metadata.get("trace_channel", None),
        "trace_start_time": metadata.get("trace_start_time", None),
        "predictions": {},
    }
    
    # Prepare merged EQCCT containers
    eqcct_probs = {}
    eqcct_picks = {}
    eqcct_errors = {}
    eqcct_tp_fp_fn = {}


    for model, name in zip(models, model_names):
        raw_probs = eutils.run_model(model, sample)  # shape (C, T)
        probs = dict(zip(model.labels, raw_probs))

        model_labels = list(model.labels)
        model_phases = [phase for phase in ["P", "S"] if phase in model_labels]
        
        pred_picks = {}
        for phase in model_phases:
            pred_picks[phase] = eutils.get_predicted_pick(probs[phase])

        errors = {}
        tp_fp_fn = {}

        for phase in model_phases:
            pred_pick = pred_picks[phase]
            true_pick = true_picks.get(phase)

            # Noise → no true picks
            if true_pick is None:
                errors[phase] = None
                tp_fp_fn[phase] = "FP" if pred_pick is not None else "TN"
                continue

            # Missed pick
            if pred_pick is None:
                errors[phase] = None
                tp_fp_fn[phase] = "FN"
                continue

            # Compute signed error in seconds
            error = (pred_pick - true_pick) / cfg.SAMPLING_RATE
            errors[phase] = error

            # Threshold determines TP vs FP
            tp_fp_fn[phase] = "TP" if abs(error) <= threshold else "FP"

        # MERGE EQCCTP and EQCCTS to EQCCT
        if name in ["EQCCTP", "EQCCTS"]:
            for phase in model_phases:
                eqcct_probs[phase] = probs[phase]
                eqcct_picks[phase] = pred_picks[phase]
                eqcct_errors[phase] = errors[phase]
                eqcct_tp_fp_fn[phase] = tp_fp_fn[phase]

        else:
            # Normal model
            obj["predictions"][name] = {
                "probs": probs,
                "picks": pred_picks,
                "errors": errors,
                "tp_fp_fn": tp_fp_fn,
            }

    # Store merged EQCCT
    if len(eqcct_probs) > 0:
        obj["predictions"]["EQCCT"] = {
            "probs": eqcct_probs,
            "picks": eqcct_picks,
            "errors": eqcct_errors,
            "tp_fp_fn": eqcct_tp_fp_fn,
        }

    return obj


def build_all_samples(generator, dataset, models, model_names, limit=None):
    """
    Build unified sample objects for an entire dataset.

    Parameters
    ----------
    dataset : list or Dataset-like
        Must support __len__ and __getitem__.
    models : list
        List of model instances.
    model_names : list
        List of model names corresponding to each model.
    limit : int or None
        Optional limit on number of samples to process.

    Returns
    -------
    list
        List of unified sample objects.
    """
    unified = []
    N = len(generator) if limit is None else min(limit, len(generator))

    for i in tqdm(range(N)):
        metadata = dataset.get_sample(i)[1]
        sample = generator[i]
        obj = build_sample_object(sample, metadata, models, model_names)
        unified.append(obj)

    return unified

In [69]:
# Load the train validation and test splits
dataset = sbd.WaveformDataset(cfg.DIR_DATA)
train_dataset, dev_dataset, test_dataset = dataset.train_dev_test()

# Preprocess dataset
train_gen, dev_gen, test_gen = tutils.preprocess_data(mutils.EQTransformerConfig(), train_dataset, dev_dataset, test_dataset)

# Preview streamlit unified object output
i = 11
sample = dev_gen[i]
sample_meta = dev_dataset.get_sample(i)[1]
models = [config.get_new_model() for config in configs.values()]
model_names = configs.keys()
obj = build_sample_object(sample, sample_meta, models, model_names)
obj["predictions"]

2026-08-27 17:10:43,397 | seisbench | WARNING | Output component order not specified, defaulting to 'ZNE'.


{'EQTransformer': {'probs': {'Detection': array([0.08806521, 0.06520062, 0.04748953, ..., 0.00543176, 0.00932268,
          0.01661367], shape=(6000,), dtype=float32),
   'P': array([2.9364731e-03, 2.2130220e-03, 1.8497488e-03, ..., 2.8988134e-05,
          9.1478294e-05, 3.6939431e-04], shape=(6000,), dtype=float32),
   'S': array([5.5771793e-04, 2.0316045e-04, 8.7065855e-05, ..., 6.0488681e-08,
          2.9791798e-07, 2.6933494e-06], shape=(6000,), dtype=float32)},
  'picks': {'P': 812, 'S': 4117},
  'errors': {'P': -5.385879999999999, 'S': 0.4966800000000012},
  'tp_fp_fn': {'P': 'FP', 'S': 'FP'}},
 'PhaseNet': {'probs': {'N': array([0.99329925, 0.99897003, 0.9991703 , ..., 0.9984043 , 0.99861646,
          0.99884844], shape=(6000,), dtype=float32),
   'P': array([6.6763698e-03, 1.0227412e-03, 8.2416448e-04, ..., 1.5629157e-05,
          8.2703882e-06, 5.5739423e-05], shape=(6000,), dtype=float32),
   'S': array([2.4401297e-05, 7.3031397e-06, 5.6142385e-06, ..., 1.5800861e-03,
   

In [70]:
# Build unified sample objects
unified_samples = build_all_samples(
    generator=dev_gen,          # or dev/test
    dataset = dev_dataset,
    models=[config.get_new_model() for config in configs.values()],
    model_names=configs.keys(),
    limit=None                      # or limit=500 for speed
)

def split_pickle(unified_samples):
    total = len(unified_samples)
    num_parts = (total + autils.CHUNK_SIZE - 1) // autils.CHUNK_SIZE

    for i in tqdm(range(num_parts)):
        start = i * autils.CHUNK_SIZE
        end = min(start + autils.CHUNK_SIZE, total)
        chunk = unified_samples[start:end]

        filename = Path(autils.DIR_VIZ_DATA) / f"{autils.FILENAME_PREFIX}_{i}.pkl"
        with open(filename, "wb") as f:
            pickle.dump(chunk, f)

# Run the split
split_pickle(unified_samples)

100%|██████████| 8/8 [00:04<00:00,  1.63it/s]


In [71]:
def load_all_samples():
    samples = []
    for pkl_file in sorted(Path(autils.DIR_VIZ_DATA).glob(f"{autils.FILENAME_PREFIX}_*.pkl")):
        with open(pkl_file, "rb") as f:
            part = pickle.load(f)
            samples.extend(part)
    return samples

samples = load_all_samples()
len(samples)

2265